### Desafio 10 — Performance

**Conceitos avaliados:** Data Skew, repartition, BroadcastHashJoin, SortMergeJoin, cache, AQE e Spark UI.

**Organização dos enunciados:** as descrições práticas ausentes foram reconstruídas e estão sinalizadas em cada seção. As perguntas e respostas originais permanecem nas células do estudante, na mesma ordem. Esta documentação não certifica a conclusão do desafio.

**Referência de estudo:** [10_performance.md](../docs/10_performance.md).

**Limites da documentação:** As descrições práticas foram reconstruídas das células atuais e conferidas com docs/10_performance.md, adicionado durante esta revisão. Esse documento registra o Desafio 10 como concluído e aprovado; isso não equivale a comprovar ganho de velocidade. O registro de Spark UI contém evidência de uma consulta, sem métricas completas e equivalentes de SortMerge nem um benchmark controlado de cache. As recomendações adicionais de docs/ não foram transformadas em novas atividades.

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("desafio_10_performance")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/18 23:46:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### Parte A — Preparação e diagnóstico

**Enunciado:** Prepare a base sintética, a dimensão de grupos e os dois experimentos de redistribuição existentes.

**Requisitos:** Use a base de 2.000.000 de IDs com oito partições e a dimensão de dez grupos. Mantenha a regra de `grupo`: IDs abaixo de 1.800.000 no grupo zero e os demais nos grupos 1–9. Inspecione contagem, schema e contagem por grupo; compare registros por partição em `repartition(8, "grupo")` e `repartition(8)`.

**Conceitos avaliados:** Data Skew e distribuição de registros.

**Origem:** enunciado prático reconstruído a partir das células originais e de [10_performance.md](../docs/10_performance.md); não é uma transcrição de um enunciado completo anterior.

In [3]:
df_base = spark.range(
    start=0,
    end=2_000_000,
    step=1,
    numPartitions=8
)

In [4]:
dados = [
    (0, "Grupo 0"),
    (1, "Grupo 1"),
    (2, "Grupo 2"),
    (3, "Grupo 3"),
    (4, "Grupo 4"),
    (5, "Grupo 5"),
    (6, "Grupo 6"),
    (7, "Grupo 7"),
    (8, "Grupo 8"),
    (9, "Grupo 9")
]

In [5]:
df_dim_grupos = spark.createDataFrame(
    dados,
    schema= ["grupo", "descricao"]
)

In [6]:
df_skew = (
    df_base.withColumn("grupo", F.when(F.col("id") < 1800000, 0).otherwise((F.col("id") % 9) + 1))
    )


In [7]:
df_skew.count()

2000000

In [8]:
df_skew.printSchema()

root
 |-- id: long (nullable = false)
 |-- grupo: long (nullable = true)



In [9]:
(
    df_skew.groupBy("grupo")
    .count()
    .orderBy("grupo")
).show()

+-----+-------+
|grupo|  count|
+-----+-------+
|    0|1800000|
|    1|  22223|
|    2|  22223|
|    3|  22222|
|    4|  22222|
|    5|  22222|
|    6|  22222|
|    7|  22222|
|    8|  22222|
|    9|  22222|
+-----+-------+



In [10]:
df_skew_repartition = df_skew.repartition(8, "grupo")

In [11]:
(
    df_skew_repartition.withColumn("quantidade_particoes", F.spark_partition_id())
    .groupBy("quantidade_particoes")
    .agg(F.count("quantidade_particoes").alias("quantidade_registros"))
    .orderBy("quantidade_particoes")
).show()

+--------------------+--------------------+
|quantidade_particoes|quantidade_registros|
+--------------------+--------------------+
|                   0|               22223|
|                   1|               22222|
|                   3|               44444|
|                   4|               66666|
|                   5|             1844445|
+--------------------+--------------------+



In [12]:
df_skew_repartition2 = df_skew.repartition(8)

In [13]:
(
    df_skew_repartition2.withColumn("quantidade_particoes", F.spark_partition_id())
    .groupBy("quantidade_particoes")
    .agg(F.count("quantidade_particoes").alias("quantidade_registros"))
    .orderBy("quantidade_particoes")
).show()

+--------------------+--------------------+
|quantidade_particoes|quantidade_registros|
+--------------------+--------------------+
|                   0|              250000|
|                   1|              250000|
|                   2|              250000|
|                   3|              250000|
|                   4|              250000|
|                   5|              250000|
|                   6|              250000|
|                   7|              250000|
+--------------------+--------------------+



Pergunta A: por que os dois DataFrames podem possuir oito partições, mas apresentar níveis de desequilíbrio completamente diferentes?

Resposta: O equilbrio ou não de dados por partições vai depender de qual chave o spark utilizara para fazer essa distribuição, no dataframe que possui a divisão desequilibrada, percebe-se que, a distribuição dos dados foi feita a partir da chave "grupo" onde temos pouco mais de 1,8 milhoes de registros representando um unico grupo, logo esses 1,8 milhoes de registros ficaram armazenados em uma partição. No segundo cenario, os dados foram reparticionados sem a utilização de uma chave especifica, nesse caso a partições ficaram perfeitamente equilibradas. Dessa forma, conclui-se que, niveis de desequilibrio dependem da chave que sera utilizada para reparticionar os dados.

#### Parte B — Comparação de estratégias de Join

**Enunciado:** Compare os joins da mesma base com a dimensão de grupos usando as duas estratégias registradas.

**Requisitos:** Use broadcast explícito no primeiro experimento e hint merge no segundo, mantendo as configurações existentes. Registre contagens, agregações por `descricao` e os planos formatted já solicitados.

**Conceitos avaliados:** estratégias de join, Exchange e Sort.

**Origem:** enunciado prático reconstruído a partir das células originais e de [10_performance.md](../docs/10_performance.md); não é uma transcrição de um enunciado completo anterior.

In [55]:
df_join_broadcast = df_skew.join(
    F.broadcast(df_dim_grupos),
    on="grupo",
    how= "inner"
)

In [56]:
df_join_broadcast.count()

2000000

In [59]:
(
    df_join_broadcast.groupBy("descricao")
    .count()
    .orderBy("descricao")
).show()

+---------+-------+
|descricao|  count|
+---------+-------+
|  Grupo 0|1800000|
|  Grupo 1|  22223|
|  Grupo 2|  22223|
|  Grupo 3|  22222|
|  Grupo 4|  22222|
|  Grupo 5|  22222|
|  Grupo 6|  22222|
|  Grupo 7|  22222|
|  Grupo 8|  22222|
|  Grupo 9|  22222|
+---------+-------+



In [17]:
df_join_broadcast.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- BroadcastHashJoin Inner BuildRight (7)
      :- Project (3)
      :  +- Filter (2)
      :     +- Range (1)
      +- BroadcastExchange (6)
         +- Filter (5)
            +- Scan ExistingRDD (4)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(8))

(2) Filter
Input [1]: [id#0L]
Condition : CASE WHEN (id#0L < 1800000) THEN true ELSE isnotnull(((id#0L % 9) + 1)) END

(3) Project
Output [2]: [id#0L, CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#6L]
Input [1]: [id#0L]

(4) Scan ExistingRDD
Output [2]: [grupo#2L, descricao#3]
Arguments: [grupo#2L, descricao#3], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(5) Filter
Input [2]: [grupo#2L, descricao#3]
Condition : isnotnull(grupo#2L)

(6) BroadcastExchange
Input [2]: [grupo#2L, descricao#3]
Arguments: HashedRelationBroadcastMode(List(input[0, bigin

In [52]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", -1)

In [53]:
df_join_sortmerge = df_skew.join(
    df_dim_grupos.hint("merge"),
    on= "grupo",
    how= "inner"
)

In [54]:
df_join_sortmerge.count()

2000000

In [21]:
(
    df_join_sortmerge.groupBy("descricao")
    .count()
    .orderBy("descricao")
).show()

+---------+-------+
|descricao|  count|
+---------+-------+
|  Grupo 0|1800000|
|  Grupo 1|  22223|
|  Grupo 2|  22223|
|  Grupo 3|  22222|
|  Grupo 4|  22222|
|  Grupo 5|  22222|
|  Grupo 6|  22222|
|  Grupo 7|  22222|
|  Grupo 8|  22222|
|  Grupo 9|  22222|
+---------+-------+



In [22]:
df_join_sortmerge.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- SortMergeJoin Inner (10)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Project (3)
      :        +- Filter (2)
      :           +- Range (1)
      +- Sort (9)
         +- Exchange (8)
            +- Filter (7)
               +- Scan ExistingRDD (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(8))

(2) Filter
Input [1]: [id#0L]
Condition : CASE WHEN (id#0L < 1800000) THEN true ELSE isnotnull(((id#0L % 9) + 1)) END

(3) Project
Output [2]: [id#0L, CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#6L]
Input [1]: [id#0L]

(4) Exchange
Input [2]: [id#0L, grupo#6L]
Arguments: hashpartitioning(grupo#6L, 200), ENSURE_REQUIREMENTS, [plan_id=1031]

(5) Sort
Input [2]: [id#0L, grupo#6L]
Arguments: [grupo#6L ASC NULLS FIRST], false, 0

(6) Scan ExistingRDD
Output [2]: [grupo#2L, descricao#3]
Arguments: [grupo#2L, descricao#3], MapPartitionsRDD[4] at applySchemaTo

Pergunta B: considerando que 90% dos registros possuem a mesma chave, qual estratégia você escolheria para esse cenário e por quê?

Resposta: Nesse cenario eu escolheria o broadcasthashjoin, como temos uma tabela menor, utilizaria a estrategia de hash por ela, dessa forma o spark replicaria uma copiada tabela para cada executor para poder gerar o join final, com isso não provocaria um shuffle bilateral, os dados não seriam redistribuidos entre partições

#### Parte C — Cache e AQE

**Enunciado:** Compare as duas consultas existentes sem cache e sobre o DataFrame cacheado.

**Requisitos:** Analise a contagem por grupo e a consulta filtrada no grupo 3. Inspecione os planos e resultados de cada versão, materialize o cache conforme a célula existente, consulte a configuração de AQE e libere a persistência ao final.

**Conceitos avaliados:** reutilização, persistência e execução adaptativa.

**Origem:** enunciado prático reconstruído a partir das células originais e de [10_performance.md](../docs/10_performance.md); não é uma transcrição de um enunciado completo anterior.

In [23]:
df_sem_cache = (
    df_skew.groupBy("grupo")
    .count()
    .orderBy("grupo")
)

In [24]:
df_sem_cache.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(8))

(2) Project
Output [1]: [CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#6L]
Input [1]: [id#0L]

(3) HashAggregate
Input [1]: [grupo#6L]
Keys [1]: [grupo#6L]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#141L]
Results [2]: [grupo#6L, count#142L]

(4) Exchange
Input [2]: [grupo#6L, count#142L]
Arguments: hashpartitioning(grupo#6L, 200), ENSURE_REQUIREMENTS, [plan_id=1056]

(5) HashAggregate
Input [2]: [grupo#6L, count#142L]
Keys [1]: [grupo#6L]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#137L]
Results [2]: [grupo#6L, count(1)#137L AS count#138L]

(6) Exchange
Input [2]: [grupo#6L, count#138L]
Arguments: rangepartitioning(grup

In [25]:
df_sem_cache.show()

+-----+-------+
|grupo|  count|
+-----+-------+
|    0|1800000|
|    1|  22223|
|    2|  22223|
|    3|  22222|
|    4|  22222|
|    5|  22222|
|    6|  22222|
|    7|  22222|
|    8|  22222|
|    9|  22222|
+-----+-------+



In [26]:
df_sem_cache2 = (
    df_skew.filter(F.col("grupo") == 3)
    .groupBy("grupo")
    .count()
    .orderBy("grupo")
)

In [27]:
df_sem_cache2.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(8))

(2) Filter
Input [1]: [id#0L]
Condition : CASE WHEN (id#0L < 1800000) THEN false ELSE (((id#0L % 9) + 1) = 3) END

(3) Project
Output [1]: [CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#6L]
Input [1]: [id#0L]

(4) HashAggregate
Input [1]: [grupo#6L]
Keys [1]: [grupo#6L]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#158L]
Results [2]: [grupo#6L, count#159L]

(5) Exchange
Input [2]: [grupo#6L, count#159L]
Arguments: hashpartitioning(grupo#6L, 200), ENSURE_REQUIREMENTS, [plan_id=1137]

(6) HashAggregate
Input [2]: [grupo#6L, count#159L]
Keys [1]: [grupo#6L]
Functions [1]: [count(1)]
Aggregate Attributes [1]: 

In [28]:
df_sem_cache2.show()

+-----+-----+
|grupo|count|
+-----+-----+
|    3|22222|
+-----+-----+



In [29]:
df_cache = df_skew.cache()

In [30]:
df_cache.count()

2000000

In [31]:
df_cache1 = (
    df_cache.groupBy("grupo")
    .count()
    .orderBy("grupo")
)

In [32]:
df_cache1.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (10)
+- Sort (9)
   +- Exchange (8)
      +- HashAggregate (7)
         +- Exchange (6)
            +- HashAggregate (5)
               +- InMemoryTableScan (1)
                     +- InMemoryRelation (2)
                           +- * Project (4)
                              +- * Range (3)


(1) InMemoryTableScan
Output [1]: [grupo#6L]
Arguments: [grupo#6L]

(2) InMemoryRelation
Arguments: [id#0L, grupo#6L], CachedRDDBuilder(org.apache.spark.sql.execution.columnar.DefaultCachedBatchSerializer@2015d911,StorageLevel(disk, memory, deserialized, 1 replicas),*(1) Project [id#0L, CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#6L]
+- *(1) Range (0, 2000000, step=1, splits=8)
,None), [id#0L ASC NULLS FIRST]

(3) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(8))

(4) Project [codegen id : 1]
Output [2]: [id#0L, CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#6L

In [33]:
df_cache1.show()

+-----+-------+
|grupo|  count|
+-----+-------+
|    0|1800000|
|    1|  22223|
|    2|  22223|
|    3|  22222|
|    4|  22222|
|    5|  22222|
|    6|  22222|
|    7|  22222|
|    8|  22222|
|    9|  22222|
+-----+-------+



In [34]:
df_cache2 = (
    df_cache.filter(F.col("grupo") == 3)
    .groupBy("grupo")
    .count()
    .orderBy("grupo")
)

In [35]:
df_cache2.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Sort (10)
   +- Exchange (9)
      +- HashAggregate (8)
         +- Exchange (7)
            +- HashAggregate (6)
               +- Filter (5)
                  +- InMemoryTableScan (1)
                        +- InMemoryRelation (2)
                              +- * Project (4)
                                 +- * Range (3)


(1) InMemoryTableScan
Output [1]: [grupo#6L]
Arguments: [grupo#6L], [isnotnull(grupo#6L), (grupo#6L = 3)]

(2) InMemoryRelation
Arguments: [id#0L, grupo#6L], CachedRDDBuilder(org.apache.spark.sql.execution.columnar.DefaultCachedBatchSerializer@2015d911,StorageLevel(disk, memory, deserialized, 1 replicas),*(1) Project [id#0L, CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#6L]
+- *(1) Range (0, 2000000, step=1, splits=8)
,None), [id#0L ASC NULLS FIRST]

(3) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(8))

(4) Project [codegen id : 1]
Output [2]: 

In [36]:
df_cache2.show()

+-----+-----+
|grupo|count|
+-----+-----+
|    3|22222|
+-----+-----+



In [37]:
spark.conf.get("spark.sql.adaptive.enabled")

'true'

In [38]:
df_cache.unpersist()

DataFrame[id: bigint, grupo: bigint]

Pergunta C: quais vantagens o cache pode oferecer nesse cenário? Por que o AQE não substitui a necessidade de analisar os custos do pipeline?

Resposta: Nesse cenario pode ser interesse trabalhar com cache() pois ele salva os dados daquela execução em memoria, entao para trabalhar com uma quantidade de registros menor o cache() custuma ser adequado, principalmente quando usamos um mesmo dataframe varias vezes no mesmo script, o ganho real dele acaba sendo evitar a recomputação do spark. Sobre o AQE não substituir a necessidade de analisar os custos de um pipeline, isso se da pelo fato do AQE ser um otimizador de execução, por exemplo ele indentifica se num cenarios especifico seria melhor o spark usar SortMergeJoin ou BroadcastHashJoin em um join. Então por mais que o spark utilize o AQE para usar uma operação ainda é importante verificar se temos um join bem modelado, e filtros bem feitos por exemplo.

#### Parte D — Diagnóstico com Spark UI

**Enunciado:** Use o registro de Spark UI da consulta de agregação da parte B para fundamentar a análise final.

**Requisitos:** Relacione os registros disponíveis aos planos e explicite as limitações dos testes realizados.

**Conceitos avaliados:** evidências e diagnóstico de execução.

**Origem:** enunciado prático reconstruído a partir das células originais e de [10_performance.md](../docs/10_performance.md); não é uma transcrição de um enunciado completo anterior.

Registro referente a consulta de agregação da Parte B:

- SQL Query ID: 23.
- Tempo total: 1 segundo.
- Jobs associados: 68, 69 e 70.

Qual estratégia foi mais adequada para os dados analisados? Quais evidências do Physical Plan e do Spark UI sustentam sua conclusão? Existe alguma limitação nos testes realizados?

Resposta: O BroadcastHashJoin foi mais adequado à estrutura dos dados: a dimensão tem apenas dez registros, permitindo evitar o shuffle da base principal para o join, mesmo com uma chave concentrando 90% dos registros. O plano mostrou BroadcastExchange e BroadcastHashJoin, o SortMerge exigiu Exchange e Sort. No Spark UI, a consulta Broadcast durou 1 segundo, e o HashAggregate observado apresentou spill zero. As limitações são o ambiente Docker local, os dados sintéticos e a ausência de métricas completas do SortMerge. Assim, a escolha é justificável, mas ainda não foi comprovado que o Broadcast foi mais rápido.

In [60]:
spark.stop()